In [1]:
!pip install mlxtend flask-ngrok pyngrok flask pandas scikit-learn numpy

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
path = '/content/drive/MyDrive/DoAn_KhaiPha'
print(os.listdir(path))

['courses_cleaned.csv', 'students_cleaned_new.csv', 'eda_overview.py']


In [4]:
# ==== CELL 2: IMPORT & LOAD DATA ====
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from flask import Flask, request, jsonify
from pyngrok import ngrok

# ── ĐỌC FILE MỚI ─────────────────────────────────────────────
df_student = pd.read_csv('/content/drive/MyDrive/DoAn_KhaiPha/students_cleaned_new.csv')
df_course  = pd.read_csv('/content/drive/MyDrive/DoAn_KhaiPha/courses_cleaned.csv')

print(f"📊 Sinh viên: {len(df_student)} dòng")
print(f"📚 Khóa học : {len(df_course)} dòng")

# ── FEATURE COLS (17 cột) ─────────────────────────────────────
FEATURE_COLS = [
    'Hours_Studied', 'Attendance', 'Previous_Scores', 'Sleep_Hours',
    'Tutoring_Sessions', 'Physical_Activity',
    'Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities',
    'Motivation_Level', 'Internet_Access', 'Family_Income',
    'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level',
    'Distance_from_Home', 'Gender',
]
print(f"✅ Feature cols: {len(FEATURE_COLS)} cột")

DIFFICULTY_MAP = {
    0: ['Beginner'],
    1: ['Beginner', 'Mixed'],
    2: ['Mixed', 'Intermediate'],
    3: ['Intermediate', 'Advanced']
}
GROUP_LABEL = {0: 'Yếu', 1: 'Trung bình', 2: 'Khá', 3: 'Giỏi'}

📊 Sinh viên: 6607 dòng
📚 Khóa học : 809 dòng
✅ Feature cols: 17 cột


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:

# ==== CELL 3: KIỂM TRA GROUP ====
# GROUP đã được tạo sẵn trong students_cleaned_new.csv
print("✅ Phân phối nhóm:")
print(df_student['GROUP_LABEL'].value_counts())
print("\nExam_Score trung bình mỗi nhóm:")
print(df_student.groupby('GROUP_LABEL')['Exam_Score'].mean().round(2))


✅ Phân phối nhóm:
GROUP_LABEL
Yếu           2131
Giỏi          1625
Trung bình    1468
Khá           1383
Name: count, dtype: int64

Exam_Score trung bình mỗi nhóm:
GROUP_LABEL
Giỏi          71.99
Khá           68.45
Trung bình    66.49
Yếu           63.33
Name: Exam_Score, dtype: float64


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [6]:

# ==== CELL 4: TRAIN KNN ====
y     = df_student['GROUP'].values
X_raw = df_student[FEATURE_COLS].values.copy().astype(float)

# Tăng trọng số các feature quan trọng nhất x3
# index 0=Hours_Studied, 1=Attendance, 2=Previous_Scores
X_raw[:, 0] *= 3   # Hours_Studied   ← quan trọng nhất
X_raw[:, 1] *= 3   # Attendance      ← quan trọng nhất
X_raw[:, 2] *= 3   # Previous_Scores ← quan trọng nhất

scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X_raw)

knn_model = KNeighborsClassifier(n_neighbors=7, metric='euclidean')
knn_model.fit(X_scaled, y)
print("✅ KNN sẵn sàng (weighted x3: Hours/Attendance/Scores)!")


✅ KNN sẵn sàng (weighted x3: Hours/Attendance/Scores)!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [7]:

# ==== CELL 5: APRIORI ====
df_course['skills_parsed'] = df_course['course_skills'].apply(ast.literal_eval)

te       = TransactionEncoder()
te_array = te.fit_transform(df_course['skills_parsed'].tolist())
df_te    = pd.DataFrame(te_array, columns=te.columns_)

frequent_itemsets = apriori(df_te, min_support=0.01, use_colnames=True)
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.20)
rules = rules[rules['lift'] >= 2.0].sort_values('lift', ascending=False).reset_index(drop=True)
print(f"✅ Apriori: {len(rules)} luật kết hợp")


def get_apriori_skills(input_skills: list, top_n=15) -> list:
    """Mở rộng kỹ năng bằng Apriori rules"""
    related = {}
    input_set = set(input_skills)
    for _, row in rules.iterrows():
        if set(row['antecedents']).issubset(input_set):
            for sk in row['consequents']:
                related[sk] = max(related.get(sk, 0), float(row['lift']))
    for sk in input_skills:
        if sk not in related:
            related[sk] = 5.0
    return [s for s, _ in sorted(related.items(), key=lambda x: x[1], reverse=True)][:top_n]


# Seed skill mặc định theo nhóm (khi sinh viên không chọn kỹ năng)
SEED_SKILLS = {
    0: ['Communication', 'Project Management', 'Microsoft Excel'],
    1: ['SQL', 'Data Analysis', 'Microsoft Excel'],
    2: ['Python Programming', 'Data Science', 'Data Analysis'],
    3: ['Machine Learning', 'Deep Learning', 'Data Science']
}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Apriori: 112 luật kết hợp


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ==== CELL 6: HÀM GỢI Ý CHÍNH ====
def recommend(encoded_vector: list, input_skills: list = None, top_n: int = 5):
    # encoded_vector: 17 số nguyên từ form sinh viên điền (đã label encode)
    # input_skills: danh sách kỹ năng sinh viên chọn (có thể bỏ trống)
    # top_n: số khóa học muốn gợi ý, mặc định 5

    # ── GIAI ĐOẠN 1: Dự đoán nhóm học lực bằng KNN ──────────────────

    # Chuyển list 17 số thành mảng numpy kiểu float để tính toán
    vec = np.array(encoded_vector, dtype=float)

    # Nhân x3 đúng như lúc train — BẮT BUỘC phải giống, không thì KNN dự đoán sai
    vec[0] *= 3   # Hours_Studied   (index 0)
    vec[1] *= 3   # Attendance      (index 1)
    vec[2] *= 3   # Previous_Scores (index 2)

    # Chuẩn hóa bằng scaler đã fit lúc train (dùng transform, KHÔNG dùng fit_transform)
    # reshape(1, -1): chuyển từ mảng 1D [17 số] → 2D [[17 số]] vì KNN yêu cầu đầu vào 2D
    x_scaled = scaler.transform(vec.reshape(1, -1))

    # KNN tìm 7 sinh viên gần nhất → trả về nhóm chiếm đa số → 0/1/2/3
    group_pred = int(knn_model.predict(x_scaled)[0])

    # Lấy danh sách độ khó khóa học phù hợp với nhóm vừa dự đoán
    # VD: nhóm 0 (Yếu) → ['Beginner'], nhóm 3 (Giỏi) → ['Intermediate','Advanced']
    difficulties = DIFFICULTY_MAP[group_pred]

    # ── GIAI ĐOẠN 2: Xác định kỹ năng ───────────────────────────────

    # Nếu sinh viên không chọn kỹ năng nào → dùng kỹ năng mặc định theo nhóm
    # VD: nhóm Giỏi → ['Machine Learning', 'Deep Learning', 'Data Science']
    if not input_skills:
        input_skills = SEED_SKILLS[group_pred]

    # Dùng Apriori mở rộng kỹ năng: từ vài kỹ năng ban đầu → tối đa 20 kỹ năng liên quan
    # VD: ['Python'] → ['Python', 'Data Science', 'SQL', 'Machine Learning', ...]
    expanded_skills = get_apriori_skills(input_skills, top_n=20)

    # ── GIAI ĐOẠN 3: Lọc và xếp hạng khóa học ───────────────────────

    # Chỉ giữ lại khóa học có độ khó nằm trong danh sách phù hợp với nhóm
    df_filtered = df_course[df_course['course_difficulty'].isin(difficulties)].copy()

    # Chuyển expanded_skills sang set để tính giao nhanh hơn
    skill_set = set(expanded_skills)

    # Đếm số kỹ năng khớp giữa từng khóa học và kỹ năng của sinh viên
    # set(x) & skill_set = phép giao hai tập hợp → lấy các kỹ năng chung
    # len(...) = đếm số kỹ năng chung → lưu vào cột 'skill_match'
    df_filtered['skill_match'] = df_filtered['skills_parsed'].apply(
        lambda x: len(set(x) & skill_set)
    )

    # Sắp xếp: ưu tiên skill_match cao nhất trước, nếu bằng nhau thì xét rating cao hơn
    # ascending=[False, False] = cả 2 đều sắp xếp giảm dần
    # .head(top_n) = lấy top N khóa tốt nhất
    top_courses = (df_filtered
                   .sort_values(by=['skill_match', 'course_rating'], ascending=[False, False])
                   .head(top_n))

    # ── GIAI ĐOẠN 4: Đóng gói kết quả ───────────────────────────────

    courses_out = []
    for _, row in top_courses.iterrows():
        # Lấy tối đa 3 kỹ năng khớp để hiển thị lý do gợi ý khóa học này
        matched = list(set(row['skills_parsed']) & skill_set)[:3]

        # Đóng gói thông tin từng khóa học thành dict để trả về JSON cho Spring Boot
        courses_out.append({
            'ten_khoa_hoc'  : str(row['course_title']),           # Tên khóa học
            'to_chuc'       : str(row['course_organization']),     # Tổ chức cung cấp
            'cap_do'        : str(row['course_difficulty']),       # Độ khó
            'diem_danh_gia' : float(row['course_rating']) if pd.notna(row['course_rating']) else 0.0,  # Rating (nếu null thì 0.0)
            'loai_chung_chi': str(row['course_certificate_type']), # Loại chứng chỉ
            'duong_dan'     : str(row['course_url']),              # Link Coursera
            'thoi_gian_hoc' : str(row['course_time']),             # Thời gian học
            'ky_nang_khop'  : matched,                             # Tối đa 3 kỹ năng khớp
            'skill_match'   : int(row['skill_match'])              # Tổng số kỹ năng khớp
        })

    # Trả về toàn bộ kết quả dạng dict → Flask jsonify → Spring Boot nhận
    return {
        'nhom_sinh_vien' : GROUP_LABEL[group_pred],  # Tên nhóm: 'Yếu'/'Trung bình'/'Khá'/'Giỏi'
        'nhom_id'        : group_pred,               # Số nhóm: 0/1/2/3
        'do_kho_phu_hop' : difficulties,             # Độ khó phù hợp với nhóm
        'ky_nang_dau_vao': input_skills,             # Kỹ năng sinh viên nhập ban đầu
        'ky_nang_mo_rong': expanded_skills[:8],      # Top 8 kỹ năng sau khi Apriori mở rộng
        'khoa_hoc_goi_y' : courses_out               # Danh sách top N khóa học gợi ý
    }

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [9]:

# ==== CELL 7: TEST THỬ 4 NHÓM ====
print("\n── Test 4 nhóm ──────────────────────────────────────────")
test_cases = [
    # (label_mong_doi, hours, attendance, prev_scores, mo_ta)
    ('Yếu',        5,  62, 55, 'Học ít, điểm thấp'),
    ('Trung bình', 15, 75, 65, 'Học trung bình'),
    ('Khá',        25, 85, 75, 'Học khá tốt'),
    ('Giỏi',       35, 95, 90, 'Học xuất sắc'),
]

for label, hours, att, prev, mo_ta in test_cases:
    # Vector mẫu: điền giá trị trung bình cho 14 feature còn lại
    vec = [hours, att, prev, 7, 1, 3, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1]
    result = recommend(vec, top_n=1)
    ok = "✅" if result['nhom_sinh_vien'] == label else "❌"
    print(f"  {ok} {mo_ta:25s} → KNN: {result['nhom_sinh_vien']:12s} | Độ khó: {result['do_kho_phu_hop']}")



── Test 4 nhóm ──────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

  ✅ Học ít, điểm thấp         → KNN: Yếu          | Độ khó: ['Beginner']
  ❌ Học trung bình            → KNN: Yếu          | Độ khó: ['Beginner']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

  ❌ Học khá tốt               → KNN: Giỏi         | Độ khó: ['Intermediate', 'Advanced']
  ✅ Học xuất sắc              → KNN: Giỏi         | Độ khó: ['Intermediate', 'Advanced']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [10]:
# ==== CELL 8: FLASK API ====
app = Flask(__name__)

@app.route('/recommend', methods=['POST'])
def api_recommend():
    """
    POST /recommend
    Body JSON:
    {
      "encoded_vector": [23, 84, 73, 7, 0, 3, 0, 2, 0, 0, 1, 0, 2, 0, 0, 2, 1],
      "input_skills"  : ["Python Programming", "Data Analysis"],
      "top_n"         : 5
    }
    """
    try:
        body           = request.get_json()
        encoded_vector = body.get('encoded_vector', [])
        input_skills   = body.get('input_skills', [])
        top_n          = int(body.get('top_n', 5))

        print(f"🔍 encoded_vector : {encoded_vector}", flush=True)
        print(f"🔍 input_skills   : {input_skills}",   flush=True)

        if len(encoded_vector) != len(FEATURE_COLS):
            return jsonify({
                'success': False,
                'error'  : f'encoded_vector cần đúng {len(FEATURE_COLS)} phần tử, nhận được {len(encoded_vector)}'
            }), 400

        result = recommend(encoded_vector, input_skills, top_n)
        print(f"✅ Kết quả: nhóm={result['nhom_sinh_vien']} | độ khó={result['do_kho_phu_hop']}", flush=True)
        return jsonify({'success': True, 'data': result})

    except Exception as e:
        print(f"❌ Lỗi: {e}", flush=True)
        return jsonify({'success': False, 'error': str(e)}), 500


@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'message': 'Model API đang chạy ✅'})


@app.route('/skills', methods=['GET'])
def list_skills():
    """Trả về top 50 kỹ năng phổ biến nhất"""
    skill_count = {}
    for lst in df_course['skills_parsed']:
        for sk in lst:
            skill_count[sk] = skill_count.get(sk, 0) + 1
    top_skills = sorted(skill_count.items(), key=lambda x: x[1], reverse=True)[:50]
    return jsonify({'skills': [s for s, _ in top_skills]})


@app.route('/feature-cols', methods=['GET'])
def feature_cols():
    """Trả về danh sách 17 feature cols theo thứ tự — Spring Boot dùng để map"""
    return jsonify({'feature_cols': FEATURE_COLS, 'total': len(FEATURE_COLS)})



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [11]:
# ==== CELL EDA: ENDPOINT /eda cho Flask API ====
# Paste cell này vào TRƯỚC cell start_server()

import io, base64, json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.0)
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 11, 'axes.labelsize': 9})

# ── Custom JSON encoder: xử lý numpy + frozenset ──────────────
class SafeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):        return int(obj)
        if isinstance(obj, (np.floating,)):       return float(obj)
        if isinstance(obj, np.ndarray):           return obj.tolist()
        if isinstance(obj, frozenset):            return list(obj)
        return super().default(obj)

def safe_jsonify(data):
    """Thay thế jsonify để tránh lỗi frozenset/numpy"""
    from flask import Response
    return Response(
        json.dumps(data, cls=SafeEncoder),
        mimetype='application/json'
    )

# ── Figure → base64 PNG ────────────────────────────────────────
def fig_to_b64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return b64

# ── EDA STUDENTS ───────────────────────────────────────────────
def eda_students():
    charts = []
    df = df_student.copy()

    grp_dist = {str(k): int(v) for k, v in df['GROUP_LABEL'].value_counts().items()}
    stats = {
        'shape':       {'rows': int(df.shape[0]), 'cols': int(df.shape[1])},
        'missing':     int(df.isnull().sum().sum()),
        'duplicates':  int(df.duplicated().sum()),
        'exam_score':  {
            'mean':   round(float(df['Exam_Score'].mean()), 2),
            'std':    round(float(df['Exam_Score'].std()),  2),
            'min':    int(df['Exam_Score'].min()),
            'max':    int(df['Exam_Score'].max()),
            'median': float(df['Exam_Score'].median()),
        },
        'group_dist':  grp_dist,
        'hours_mean':  round(float(df['Hours_Studied'].mean()), 1),
        'attend_mean': round(float(df['Attendance'].mean()), 1),
    }

    # Biểu đồ 1: Exam Score
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Phân phối Exam Score', fontweight='bold')
    axes[0].hist(df['Exam_Score'], bins=30, color='teal', edgecolor='white', density=True, alpha=0.8)
    df['Exam_Score'].plot.kde(ax=axes[0], color='darkred', linewidth=2)
    axes[0].axvline(df['Exam_Score'].mean(), color='navy', linestyle='--',
                    label=f"Mean={df['Exam_Score'].mean():.1f}")
    axes[0].set_title('Histogram + KDE'); axes[0].legend(fontsize=8)
    order = [o for o in ['Yếu','Trung bình','Khá','Giỏi'] if o in df['GROUP_LABEL'].unique()]
    sns.boxplot(data=df, x='GROUP_LABEL', y='Exam_Score', palette='Set1', order=order, ax=axes[1])
    axes[1].set_title('Exam Score theo Nhóm'); axes[1].set_xlabel('Nhóm học lực')
    plt.tight_layout()
    charts.append({'title': 'Phân phối Exam Score', 'image': fig_to_b64(fig)})

    # Biểu đồ 2: Biến liên tục
    cont = ['Hours_Studied','Attendance','Previous_Scores','Sleep_Hours','Tutoring_Sessions','Physical_Activity']
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    fig.suptitle('Phân phối các biến liên tục', fontweight='bold')
    for ax, feat in zip(axes.flatten(), cont):
        data = df[feat].dropna()
        ax.hist(data, bins=20, edgecolor='white', color='steelblue', alpha=0.8, density=True)
        data.plot.kde(ax=ax, color='darkred', linewidth=1.5)
        ax.axvline(data.mean(), color='orange', linestyle='--', linewidth=1.2)
        ax.set_title(feat.replace('_',' '), fontsize=9)
    plt.tight_layout()
    charts.append({'title': 'Phân phối biến liên tục', 'image': fig_to_b64(fig)})

    # Biểu đồ 3: Tương quan
    fig, ax = plt.subplots(figsize=(12, 9))
    corr = df.select_dtypes(include='number').corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
                center=0, linewidths=0.4, ax=ax, annot_kws={'size': 7})
    ax.set_title('Ma trận tương quan', fontsize=12, fontweight='bold')
    plt.tight_layout()
    charts.append({'title': 'Ma trận tương quan', 'image': fig_to_b64(fig)})

    # Biểu đồ 4: Top features
    corr_target = (df.select_dtypes(include='number')
                   .corr()['Exam_Score'].drop('Exam_Score')
                   .sort_values(key=abs, ascending=False).head(10))
    fig, ax = plt.subplots(figsize=(9, 5))
    colors_bar = ['tomato' if v < 0 else 'steelblue' for v in corr_target]
    corr_target.plot(kind='barh', ax=ax, color=colors_bar, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title('Top 10 features tương quan với Exam_Score', fontweight='bold')
    plt.tight_layout()
    charts.append({'title': 'Tương quan với Exam Score', 'image': fig_to_b64(fig)})

    return {'stats': stats, 'charts': charts}

# ── EDA COURSES ────────────────────────────────────────────────
def eda_courses():
    charts = []
    df = df_course.drop(columns=["skills_parsed"], errors="ignore").copy()
    df['course_rating']            = pd.to_numeric(df['course_rating'],            errors='coerce')
    df['course_students_enrolled'] = pd.to_numeric(df['course_students_enrolled'], errors='coerce')
    df['course_reviews_num']       = pd.to_numeric(df['course_reviews_num'],       errors='coerce')

    stats = {
        'shape':           {'rows': int(df.shape[0]), 'cols': int(df.shape[1])},
        'missing':         int(df.isnull().sum().sum()),
        'duplicates':      int(df.duplicated().sum()),
        'rating':          {
            'mean':   round(float(df['course_rating'].mean()), 2),
            'min':    float(df['course_rating'].min()),
            'max':    float(df['course_rating'].max()),
            'median': float(df['course_rating'].median()),
        },
        'enrolled_median': int(df['course_students_enrolled'].median()),
        'top_org':         str(df['course_organization'].value_counts().idxmax()),
        'top_org_count':   int(df['course_organization'].value_counts().max()),
        'difficulty_dist': {str(k): int(v) for k, v in df['course_difficulty'].value_counts().items()},
        'cert_dist':       {str(k): int(v) for k, v in df['course_certificate_type'].value_counts().items()},
    }

    # Biểu đồ 1: Rating + Enrolled + Reviews
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle('Phân phối biến số — Courses', fontweight='bold')
    rating = df['course_rating'].dropna()
    axes[0].hist(rating, bins=20, color='steelblue', edgecolor='white')
    axes[0].axvline(rating.mean(), color='red', linestyle='--', label=f"Mean={rating.mean():.2f}")
    axes[0].set_title('Course Rating'); axes[0].legend(fontsize=8)
    axes[1].hist(np.log1p(df['course_students_enrolled'].dropna()), bins=25, color='coral', edgecolor='white')
    axes[1].set_title('Số học viên (log scale)'); axes[1].set_xlabel('log(students+1)')
    axes[2].hist(np.log1p(df['course_reviews_num'].dropna()), bins=25, color='mediumpurple', edgecolor='white')
    axes[2].set_title('Số lượt đánh giá (log scale)'); axes[2].set_xlabel('log(reviews+1)')
    plt.tight_layout()
    charts.append({'title': 'Phân phối biến số', 'image': fig_to_b64(fig)})

    # Biểu đồ 2: Biến phân loại
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('Phân phối biến phân loại — Courses', fontweight='bold')
    for ax, feat, color in zip(axes,
        ['course_certificate_type','course_difficulty','course_organization'],
        ['#4C72B0','#55A868','#C44E52']):
        df[feat].value_counts().head(10).plot(kind='barh', ax=ax, color=color, edgecolor='white')
        ax.set_title(feat.replace('course_','').replace('_',' ').title(), fontsize=9)
        ax.set_xlabel('Số lượng')
    plt.tight_layout()
    charts.append({'title': 'Phân phối biến phân loại', 'image': fig_to_b64(fig)})

    # Biểu đồ 3: Boxplot
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('So sánh nhóm — Courses', fontweight='bold')
    sns.boxplot(data=df, x='course_difficulty', y='course_rating', palette='Set2', ax=axes[0])
    axes[0].set_title('Rating theo Độ khó')
    sns.boxplot(data=df, x='course_certificate_type',
                y=np.log1p(df['course_students_enrolled']), palette='Set3', ax=axes[1])
    axes[1].set_title('Học viên (log) theo Chứng chỉ')
    axes[1].tick_params(axis='x', rotation=15)
    plt.tight_layout()
    charts.append({'title': 'So sánh nhóm', 'image': fig_to_b64(fig)})

    return {'stats': stats, 'charts': charts}

# ── Flask route /eda ───────────────────────────────────────────
@app.route('/eda', methods=['GET'])
def api_eda():
    dataset = request.args.get('dataset', 'students').lower()
    try:
        if dataset == 'students':
            result = eda_students()
        elif dataset == 'courses':
            result = eda_courses()
        else:
            return safe_jsonify({'success': False, 'error': 'dataset phải là students hoặc courses'})
        return safe_jsonify({'success': True, 'dataset': dataset, 'data': result})
    except Exception as e:
        import traceback
        print(f"❌ EDA lỗi: {traceback.format_exc()}", flush=True)
        return safe_jsonify({'success': False, 'error': str(e)})

print("✅ Route /eda đã đăng ký!")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Route /eda đã đăng ký!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:

# ==== CELL 9: CHẠY SERVER ====
def start_server(port=5000):
    ngrok.set_auth_token("3Dvkw47jGAYuZFKgaCWOg6zTlDh_7CEm5LhAzUVA5DR6KQXdy")
    public_url = ngrok.connect(port)
    print(f"\n🚀 API: {public_url}")
    print(f"   POST {public_url}/recommend")
    print(f"   GET  {public_url}/health")
    print(f"   GET  {public_url}/skills")
    print(f"   GET  {public_url}/feature-cols")
    app.run(port=port)

start_server(port=5000)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


🚀 API: NgrokTunnel: "https://glitzy-confusion-casualty.ngrok-free.dev" -> "http://localhost:5000"
   POST NgrokTunnel: "https://glitzy-confusion-casualty.ngrok-free.dev" -> "http://localhost:5000"/recommend
   GET  NgrokTunnel: "https://glitzy-confusion-casualty.ngrok-free.dev" -> "http://localhost:5000"/health
   GET  NgrokTunnel: "https://glitzy-confusion-casualty.ngrok-free.dev" -> "http://localhost:5000"/skills
   GET  NgrokTunnel: "https://glitzy-confusion-casualty.ngrok-free.dev" -> "http://localhost:5000"/feature-cols
 * Serving Flask app '__main__'


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

 * Debug mode: off


Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac